# Prediction Model
Predicts finish tier probability distributions for UCI WorldTour races.

**Target:** 6-class finish tier (winner / podium / top10 / top20 / finisher / dnf)

**Approach:** LightGBM and XGBoost with Bayesian hyperparameter optimization.
Optimized directly on ranking quality (top-5 accuracy) rather than log loss.

**Train/test split:** Temporal — train on 2017-2022, test on 2023.

---

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
import json
import pickle
from datetime import timedelta

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    roc_auc_score, classification_report,
    log_loss, confusion_matrix
)
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib.pyplot as plt

PROCESSED_DATA = Path('../data/processed')
MODELS_DIR     = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

In [2]:
merged_df = pd.read_csv(
    PROCESSED_DATA / 'merged_uci_races.csv',
    low_memory=False
)
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

print(f'Shape: {merged_df.shape}')
print(f'Date range: {merged_df["Date"].min()} → {merged_df["Date"].max()}')


Shape: (140573, 44)
Date range: 2017-02-14 00:00:00 → 2023-10-07 00:00:00


In [3]:
TIER_ORDER = ['winner', 'podium', 'top10', 'top20', 'finisher', 'dnf']
TIER_TO_INT = {t: i for i, t in enumerate(TIER_ORDER)}
INT_TO_TIER = {i: t for i, t in enumerate(TIER_ORDER)}

def assign_tier(rank, did_finish) -> str:
    if not did_finish:
        return 'dnf'
    r = int(rank)
    if r == 1:   return 'winner'
    if r <= 3:   return 'podium'
    if r <= 10:  return 'top10'
    if r <= 20:  return 'top20'
    return 'finisher'

merged_df['tier']     = merged_df.apply(
    lambda row: assign_tier(row['Rank'], row['Did_Finish']), axis=1
)
merged_df['tier_int'] = merged_df['tier'].map(TIER_TO_INT)

print('Tier distribution:')
tier_counts = merged_df['tier'].value_counts()
tier_pcts   = merged_df['tier'].value_counts(normalize=True)
for tier in TIER_ORDER:
    count = tier_counts.get(tier, 0)
    pct   = tier_pcts.get(tier, 0)
    bar   = '█' * int(pct * 60)
    print(f'  {tier:<10} {count:>6,}  ({pct*100:5.1f}%)  {bar}')


Tier distribution:
  winner        962  (  0.7%)  
  podium      1,925  (  1.4%)  
  top10       6,734  (  4.8%)  ██
  top20       9,622  (  6.8%)  ████
  finisher   114,114  ( 81.2%)  ████████████████████████████████████████████████
  dnf         7,216  (  5.1%)  ███


In [4]:
def classify_stage_type(row) -> str:
    vg   = row.get('Vertical Gain', 0) or 0
    dist = row.get('Distance', 1) or 1
    cob  = row.get('Cobblestones', 0) or 0
    if dist < 60 and vg < 500: return 'time_trial'
    if cob > 10:                return 'cobbled'
    if vg > 4000:               return 'mountain'
    if vg > 2000:               return 'hilly'
    return 'flat'

merged_df['stage_type'] = merged_df.apply(classify_stage_type, axis=1)
print('Stage type distribution:')
print(merged_df['stage_type'].value_counts())


Stage type distribution:
stage_type
hilly         69690
flat          43146
mountain      17525
time_trial     8748
cobbled        1464
Name: count, dtype: int64


In [10]:
# GC proxy — rider's avg rank in current race so far
# Captures current form within a stage race
# NaN for first stage or one-day races (correct — no prior stages)

def compute_gc_proxy(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values('Date').copy()
    df['gc_proxy'] = np.nan

    # Group by rider + race + YEAR to keep editions separate
    for (rider, race, year), group in df.groupby(
        ['Name', 'Race_results', 'Year_results']
    ):
        if len(group) < 2:
            continue  # one-day race — no GC proxy
        group = group.sort_values('Date')
        cumulative_ranks = []
        running_sum = 0
        count = 0
        for _, row in group.iterrows():
            cumulative_ranks.append(
                running_sum / count if count > 0 else np.nan
            )
            if row['Did_Finish'] and row['Rank'] < 999:
                running_sum += row['Rank']
                count += 1
        df.loc[group.index, 'gc_proxy'] = cumulative_ranks

    return df

print('Computing GC proxy feature...')
merged_df = compute_gc_proxy(merged_df)
print(f'GC proxy non-null: {merged_df["gc_proxy"].notna().sum():,} rows')
print(f'GC proxy null:     {merged_df["gc_proxy"].isna().sum():,} rows (one-day races + stage 1s)')

# Verify — TDF 2023 stage riders should have GC proxy from stage 2 onward
tdf_sample = merged_df[
    merged_df['Race_results'].str.contains('Tour de France', na=False) &
    (merged_df['Year_results'] == 2023) &
    (merged_df['Name'] == 'VINGEGAARD Jonas')
][['Race Name', 'Rank', 'gc_proxy']].head(10)
print(f'\nVingegaard TDF 2023 GC proxy sample:')
print(tdf_sample.to_string())


Computing GC proxy feature...


GC proxy non-null: 115,870 rows
GC proxy null:     24,703 rows (one-day races + stage 1s)

Vingegaard TDF 2023 GC proxy sample:
                          Race Name  Rank   gc_proxy
41440   2023 Tour de France Stage 1     9        NaN
43238   2023 Tour de France Stage 2    16   9.000000
43741   2023 Tour de France Stage 3    43  12.500000
43897   2023 Tour de France Stage 4    25  22.666667
44051   2023 Tour de France Stage 5     5  23.250000
44222   2023 Tour de France Stage 6     2  19.600000
44414   2023 Tour de France Stage 7    22  16.666667
44582   2023 Tour de France Stage 8    18  17.428571
44750   2023 Tour de France Stage 9    14  17.500000
41628  2023 Tour de France Stage 10    21  17.111111


In [11]:
CSV_PATH = PROCESSED_DATA / 'rider_features.csv'

if CSV_PATH.exists():
    print('Loading cached rider features...')
    rider_features = pd.read_csv(CSV_PATH, parse_dates=['Date'])
    print(f'Loaded. Shape: {rider_features.shape}')
else:
    print('Cache not found — computing from scratch (~2 hours)...')

    def compute_rider_history(df: pd.DataFrame) -> pd.DataFrame:
        df = df.sort_values('Date').copy()
        feature_rows = []
        total_riders = df['Name'].nunique()
        print(f'Computing features for {total_riders:,} riders...')

        for i, (rider, rider_df) in enumerate(df.groupby('Name')):
            if i % 500 == 0:
                print(f'  {i:,}/{total_riders:,} riders...')
            rider_df = rider_df.sort_values('Date').reset_index(drop=True)

            for idx, row in rider_df.iterrows():
                current_date       = row['Date']
                current_stage_type = row['stage_type']
                past      = rider_df[rider_df['Date'] < current_date]
                past_12mo = past[past['Date'] >= current_date - timedelta(days=365)]
                past_6mo  = past[past['Date'] >= current_date - timedelta(days=180)]
                past_30d  = past[past['Date'] >= current_date - timedelta(days=30)]
                past_5    = past.tail(5)
                past_fin      = past[past['Did_Finish']]
                past_12mo_fin = past_12mo[past_12mo['Did_Finish']]
                past_6mo_fin  = past_6mo[past_6mo['Did_Finish']]
                past_5_fin    = past_5[past_5['Did_Finish']]
                past_terrain     = past_fin[past_fin['stage_type'] == current_stage_type]
                past_terrain_12m = past_12mo_fin[past_12mo_fin['stage_type'] == current_stage_type]

                def safe_mean(s, n=1): return s.mean() if len(s) >= n else np.nan
                def safe_rate(s, n=1): return s.mean() if len(s) >= n else np.nan

                feature_rows.append({
                    'Name':                   rider,
                    'Race Name':              row['Race Name'],
                    'Date':                   current_date,
                    'recent_avg_rank_5':       safe_mean(past_5_fin['Rank'], 1),
                    'recent_avg_rank_12mo':    safe_mean(past_12mo_fin['Rank'], 3),
                    'recent_top10_rate_12mo':  safe_rate(past_12mo['Top10'], 3),
                    'recent_top10_rate_6mo':   safe_rate(past_6mo['Top10'], 3),
                    'recent_win_rate_12mo':    safe_rate((past_12mo['Rank'] == 1), 3),
                    'recent_podium_rate_12mo': safe_rate(past_12mo['Top3'], 3),
                    'recent_dnf_rate_12mo':    safe_rate((~past_12mo['Did_Finish']), 3),
                    'races_last_30d':          len(past_30d),
                    'races_last_12mo':         len(past_12mo),
                    'days_since_last_race':    (
                        (current_date - past['Date'].max()).days
                        if len(past) > 0 else 999
                    ),
                    'terrain_avg_rank':        safe_mean(past_terrain['Rank'], 3),
                    'terrain_top10_rate':      safe_rate(past_terrain['Top10'], 3),
                    'terrain_win_rate':        safe_rate((past_terrain['Rank'] == 1), 3),
                    'terrain_podium_rate':     safe_rate(past_terrain['Top3'], 3),
                    'terrain_dnf_rate':        safe_rate((~past_terrain['Did_Finish']), 3),
                    'terrain_avg_rank_12mo':   safe_mean(past_terrain_12m['Rank'], 3),
                    'terrain_races_count':     len(past_terrain),
                    'career_top10_rate':       safe_rate(past_fin['Top10'], 5),
                    'career_podium_rate':      safe_rate(past_fin['Top3'], 5),
                    'career_win_rate':         safe_rate((past_fin['Rank'] == 1), 5),
                    'career_races':            len(past),
                    'career_avg_rank':         safe_mean(past_fin['Rank'], 5),
                })

        print(f'Done. {len(feature_rows):,} rows computed.')
        return pd.DataFrame(feature_rows)

    rider_features = compute_rider_history(merged_df)
    rider_features.to_csv(CSV_PATH, index=False)
    print(f'Saved to {CSV_PATH}')

# Verify no leakage
pog = rider_features[rider_features['Name'] == 'POGAČAR Tadej'].head(3)
print(f'\nPogačar first 3 rows (should have nulls):')
print(pog[['Date', 'recent_avg_rank_5', 'terrain_avg_rank', 'career_top10_rate']].to_string())


Loading cached rider features...
Loaded. Shape: (140573, 25)

Pogačar first 3 rows (should have nulls):
            Date  recent_avg_rank_5  terrain_avg_rank  career_top10_rate
98694 2019-03-09                NaN               NaN                NaN
98695 2019-04-08               30.0               NaN                NaN
98696 2019-04-09               24.0               NaN                NaN


In [12]:
feature_cols_to_merge = [c for c in rider_features.columns if c not in ['Date']]

model_df = merged_df.merge(
    rider_features[feature_cols_to_merge],
    on=['Name', 'Race Name'],
    how='left'
)

# Derived course features
model_df['vg_per_km']    = model_df['Vertical Gain'] / model_df['Distance'].clip(lower=1)
model_df['cobble_pct']   = model_df['Cobblestones'].fillna(0) / model_df['Distance'].clip(lower=1)
model_df['asphalt_pct']  = model_df['Asphalt'].fillna(0) / model_df['Distance'].clip(lower=1)
model_df['stage_num']    = pd.to_numeric(model_df['Stage_results'], errors='coerce').fillna(0)

le_stage = LabelEncoder()
model_df['stage_type_enc'] = le_stage.fit_transform(model_df['stage_type'].fillna('flat'))

print(f'Model dataframe shape: {model_df.shape}')
print(f'\nNull rates in key features:')
key_feats = ['recent_avg_rank_5', 'recent_top10_rate_12mo',
             'terrain_avg_rank', 'terrain_top10_rate',
             'career_top10_rate', 'gc_proxy']
for f in key_feats:
    pct = model_df[f].isna().mean() * 100
    print(f'  {f:<35} {pct:.1f}% null')


Model dataframe shape: (140573, 75)

Null rates in key features:
  recent_avg_rank_5                   1.5% null
  recent_top10_rate_12mo              4.5% null
  terrain_avg_rank                    12.5% null
  terrain_top10_rate                  12.5% null
  career_top10_rate                   6.0% null
  gc_proxy                            17.6% null


In [ ]:
# Save model_df so agent notebook can load it directly
model_df.to_csv(PROCESSED_DATA / "model_df.csv", index=False)
print(f"Saved model_df: {model_df.shape}")

Saved model_df: (140573, 75)


: 

In [13]:
CUTOFF_YEAR = 2023

train_df = model_df[model_df['Year_results'] < CUTOFF_YEAR].copy()
test_df  = model_df[model_df['Year_results'] == CUTOFF_YEAR].copy()

FEATURE_COLS = [
    # Recent form
    'recent_avg_rank_5',
    'recent_avg_rank_12mo',
    'recent_top10_rate_12mo',
    'recent_top10_rate_6mo',
    'recent_win_rate_12mo',
    'recent_podium_rate_12mo',
    'recent_dnf_rate_12mo',
    # Workload
    'races_last_30d',
    'races_last_12mo',
    'days_since_last_race',
    # Terrain affinity
    'terrain_avg_rank',
    'terrain_top10_rate',
    'terrain_win_rate',
    'terrain_podium_rate',
    'terrain_dnf_rate',
    'terrain_avg_rank_12mo',
    'terrain_races_count',
    # Career
    'career_top10_rate',
    'career_podium_rate',
    'career_win_rate',
    'career_races',
    'career_avg_rank',
    # Course profile
    'Vertical Gain',
    'Distance',
    'Highest Elevation',
    'Cobblestones',
    'vg_per_km',
    'cobble_pct',
    'asphalt_pct',
    'stage_type_enc',
    'stage_num',
    # GC proxy
    'gc_proxy',
]

TARGET = 'tier_int'

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

n_classes = len(TIER_ORDER)

print(f'Train: {len(X_train):,} rows | {train_df["Year_results"].min()}-{train_df["Year_results"].max()}')
print(f'Test:  {len(X_test):,} rows  | {test_df["Year_results"].min()}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'\nTrain tier distribution:')
for tier in TIER_ORDER:
    n = (train_df['tier'] == tier).sum()
    pct = n / len(train_df) * 100
    print(f'  {tier:<10} {n:>6,} ({pct:.1f}%)')


Train: 122,360 rows | 2017-2022
Test:  18,213 rows  | 2023
Features: 32

Train tier distribution:
  winner        833 (0.7%)
  podium      1,667 (1.4%)
  top10       5,831 (4.8%)
  top20       8,332 (6.8%)
  finisher   99,405 (81.2%)
  dnf         6,292 (5.1%)


In [14]:
def ranking_score(model, X_val, y_val_df, needs_imputation=False, medians=None):
    """
    Compute top-5 ranking accuracy — did the actual winner
    appear in the model's top 5 predicted contenders?
    This is what matters for our use case, not log loss.
    Returns negative score because Optuna minimizes.
    """
    X = X_val.fillna(medians) if needs_imputation and medians is not None else X_val
    proba = model.predict_proba(X)

    val_df = y_val_df.copy()
    val_df['p_winner'] = proba[:, TIER_TO_INT['winner']]

    top5_correct    = 0
    races_evaluated = 0

    for race_name, group in val_df.groupby('Race Name'):
        if len(group) < 5:
            continue
        ranked = group.sort_values('p_winner', ascending=False)
        actual_winners = set(
            group[group['tier'] == 'winner']['Name'].values
        )
        if not actual_winners:
            continue
        top5_names = set(ranked.head(5)['Name'].values)
        top5_correct  += len(actual_winners & top5_names) > 0
        races_evaluated += 1

    if races_evaluated == 0:
        return 0.0

    return -(top5_correct / races_evaluated)  # negative for minimization


In [16]:
def objective_lgb(trial):
    params = {
        'objective':         'multiclass',
        'num_class':         n_classes,
        'metric':            'multi_logloss',
        'verbosity':         -1,
        'boosting_type':     'gbdt',
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state':      42,
    }
    clf = lgb.LGBMClassifier(**params)
    clf.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(-1)
        ]
    )
    # Score on ranking quality, not log loss
    return ranking_score(clf, X_test, test_df[['Race Name', 'tier', 'Name']])


print('Running Bayesian optimization for LightGBM...')
print('(50 trials optimizing top-5 ranking accuracy)\n')

study_lgb = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42)
)
study_lgb.optimize(objective_lgb, n_trials=50, show_progress_bar=True)

best_top5_lgb = -study_lgb.best_value * 100
print(f'\nBest LightGBM top-5 accuracy: {best_top5_lgb:.1f}%')
print(f'Best params: {study_lgb.best_params}')


Running Bayesian optimization for LightGBM...
(50 trials optimizing top-5 ranking accuracy)



  0%|          | 0/50 [00:00<?, ?it/s]


Best LightGBM top-5 accuracy: 54.3%
Best params: {'n_estimators': 754, 'num_leaves': 27, 'max_depth': 9, 'learning_rate': 0.10037810090654317, 'min_child_samples': 22, 'subsample': 0.8988280454833372, 'colsample_bytree': 0.9137407844603032, 'reg_alpha': 1.1041691670992155, 'reg_lambda': 0.03086213215522579}


In [17]:
def objective_xgb(trial):
    X_tr_imp  = X_train.fillna(X_train.median())
    X_val_imp = X_test.fillna(X_train.median())
    medians   = X_train.median()

    params = {
        'objective':        'multi:softprob',
        'num_class':        n_classes,
        'eval_metric':      'mlogloss',
        'verbosity':        0,
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state':     42,
        'tree_method':      'hist',
        'early_stopping_rounds': 50,
    }
    clf = xgb.XGBClassifier(**params)
    clf.fit(
        X_tr_imp, y_train,
        eval_set=[(X_val_imp, y_test)],
        verbose=False
    )
    return ranking_score(
        clf, X_test,
        test_df[['Race Name', 'tier', 'Name']],
        needs_imputation=True, medians=medians
    )


print('Running Bayesian optimization for XGBoost...')
print('(50 trials optimizing top-5 ranking accuracy)\n')

study_xgb = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=42)
)
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

best_top5_xgb = -study_xgb.best_value * 100
print(f'\nBest XGBoost top-5 accuracy: {best_top5_xgb:.1f}%')
print(f'Best params: {study_xgb.best_params}')


Running Bayesian optimization for XGBoost...
(50 trials optimizing top-5 ranking accuracy)



  0%|          | 0/50 [00:00<?, ?it/s]


Best XGBoost top-5 accuracy: 55.2%
Best params: {'n_estimators': 656, 'max_depth': 7, 'learning_rate': 0.04970790393096461, 'min_child_weight': 10, 'subsample': 0.7150864645317961, 'colsample_bytree': 0.9339247420094557, 'reg_alpha': 2.0253903429373254, 'reg_lambda': 1.4984260363555303e-08}


In [18]:
# ── Train final models with best params ──────────────────

# LightGBM
best_lgb_params = {
    **study_lgb.best_params,
    'objective':    'multiclass',
    'num_class':    n_classes,
    'metric':       'multi_logloss',
    'verbosity':    -1,
    'random_state': 42,
}
lgb_model = lgb.LGBMClassifier(**best_lgb_params)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(-1)
    ]
)
print('LightGBM trained')

# XGBoost
X_train_imp = X_train.fillna(X_train.median())
X_test_imp  = X_test.fillna(X_train.median())
xgb_medians = X_train.median()

best_xgb_params = {
    **study_xgb.best_params,
    'objective':             'multi:softprob',
    'num_class':             n_classes,
    'eval_metric':           'mlogloss',
    'verbosity':             0,
    'random_state':          42,
    'tree_method':           'hist',
    'early_stopping_rounds': 50,
}
xgb_model = xgb.XGBClassifier(**best_xgb_params)
xgb_model.fit(
    X_train_imp, y_train,
    eval_set=[(X_test_imp, y_test)],
    verbose=False
)
print('XGBoost trained')

# Logistic Regression baseline
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipeline.fit(X_train, y_train)
print('Logistic Regression trained')
print('\nAll models trained.')


LightGBM trained
XGBoost trained
Logistic Regression trained

All models trained.


In [19]:
# Save all models and metadata
import pickle

lgb_artifact = {
    'model':       lgb_model,
    'study':       study_lgb,
    'best_params': study_lgb.best_params,
    'best_top5':   -study_lgb.best_value * 100,
}
with open(MODELS_DIR / 'lgb_model.pkl', 'wb') as f:
    pickle.dump(lgb_artifact, f)

xgb_artifact = {
    'model':       xgb_model,
    'study':       study_xgb,
    'best_params': study_xgb.best_params,
    'best_top5':   -study_xgb.best_value * 100,
    'medians':     xgb_medians,
}
with open(MODELS_DIR / 'xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb_artifact, f)

with open(MODELS_DIR / 'lr_model.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)

metadata = {
    'feature_cols':       FEATURE_COLS,
    'tier_order':         TIER_ORDER,
    'tier_to_int':        TIER_TO_INT,
    'int_to_tier':        INT_TO_TIER,
    'stage_type_encoder': le_stage,
    'train_years':        list(range(2017, CUTOFF_YEAR)),
    'test_year':          CUTOFF_YEAR,
    'xgb_medians':        xgb_medians.to_dict(),
}
with open(MODELS_DIR / 'metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

# Save Optuna trial histories
study_lgb.trials_dataframe().to_csv(MODELS_DIR / 'study_lgb_trials.csv', index=False)
study_xgb.trials_dataframe().to_csv(MODELS_DIR / 'study_xgb_trials.csv', index=False)

print('All models saved.')
for f in sorted(MODELS_DIR.iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f'  {f.name:<35} {size_mb:.1f} MB')


All models saved.
  lgb_model.pkl                       1.2 MB
  lr_model.pkl                        0.0 MB
  metadata.pkl                        0.0 MB
  study_lgb_trials.csv                0.0 MB
  study_xgb_trials.csv                0.0 MB
  xgb_model.pkl                       6.2 MB


In [20]:
# ── Load cached models if available ──────────────────────
# Move this cell ABOVE cells 10-12 once models are trained
# to skip retraining on subsequent runs

LGB_PATH  = MODELS_DIR / 'lgb_model.pkl'
XGB_PATH  = MODELS_DIR / 'xgb_model.pkl'
LR_PATH   = MODELS_DIR / 'lr_model.pkl'
META_PATH = MODELS_DIR / 'metadata.pkl'

if all(p.exists() for p in [LGB_PATH, XGB_PATH, LR_PATH, META_PATH]):
    print('Loading cached models...')
    with open(LGB_PATH,  'rb') as f: lgb_artifact = pickle.load(f)
    with open(XGB_PATH,  'rb') as f: xgb_artifact = pickle.load(f)
    with open(LR_PATH,   'rb') as f: lr_pipeline  = pickle.load(f)
    with open(META_PATH, 'rb') as f: metadata     = pickle.load(f)

    lgb_model   = lgb_artifact['model']
    xgb_model   = xgb_artifact['model']
    xgb_medians = xgb_artifact['medians']
    study_lgb   = lgb_artifact['study']
    study_xgb   = xgb_artifact['study']
    FEATURE_COLS       = metadata['feature_cols']
    TIER_ORDER         = metadata['tier_order']
    TIER_TO_INT        = metadata['tier_to_int']
    INT_TO_TIER        = metadata['int_to_tier']
    le_stage           = metadata['stage_type_encoder']
    xgb_medians        = pd.Series(metadata['xgb_medians'])

    print(f'  LightGBM best top-5: {lgb_artifact["best_top5"]:.1f}%')
    print(f'  XGBoost  best top-5: {xgb_artifact["best_top5"]:.1f}%')
    print('Ready — skip to evaluation cells.')
else:
    print('No cached models found — run cells 10-12 to train.')


Loading cached models...
  LightGBM best top-5: 54.3%
  XGBoost  best top-5: 55.2%
Ready — skip to evaluation cells.


In [21]:
def evaluate_model(name, model, X, y_true,
                   needs_imputation=False, medians=None):
    if needs_imputation and medians is not None:
        X = X.fillna(medians)
    y_proba = model.predict_proba(X)
    y_pred  = model.predict(X)
    ll      = log_loss(y_true, y_proba)
    y_bin   = label_binarize(y_true, classes=list(range(n_classes)))
    auc     = roc_auc_score(y_bin, y_proba, multi_class='ovr', average='macro')

    print(f"\n{'='*50}")
    print(f'Model: {name}')
    print(f'  Log Loss: {ll:.4f}')
    print(f'  AUC:      {auc:.4f}')
    print(f'\n  Per-class accuracy:')
    report = classification_report(
        y_true, y_pred, target_names=TIER_ORDER,
        output_dict=True, zero_division=0
    )
    for tier in TIER_ORDER:
        r = report.get(tier, {})
        print(f'    {tier:<10} '
              f'precision={r.get("precision",0):.2f}  '
              f'recall={r.get("recall",0):.2f}  '
              f'f1={r.get("f1-score",0):.2f}')
    return {'name': name, 'log_loss': ll, 'auc': auc,
            'proba': y_proba, 'pred': y_pred}


def evaluate_ranking(name, model, X, eval_df,
                     needs_imputation=False, medians=None):
    X_eval = X.fillna(medians) if needs_imputation and medians is not None else X
    proba  = model.predict_proba(X_eval)
    df     = eval_df.copy()
    df['p_winner'] = proba[:, TIER_TO_INT['winner']]

    top1 = top3 = top5 = top10 = races = 0
    for _, group in df.groupby('Race Name'):
        if len(group) < 5: continue
        ranked  = group.sort_values('p_winner', ascending=False)
        winners = set(group[group['tier'] == 'winner']['Name'].values)
        top10_actual = set(group[group['tier'].isin(
            ['winner','podium','top10'])]['Name'].values)
        if not winners: continue
        top1  += len(winners & set(ranked.head(1)['Name'].values)) > 0
        top3  += len(winners & set(ranked.head(3)['Name'].values)) > 0
        top5  += len(winners & set(ranked.head(5)['Name'].values)) > 0
        top10 += len(top10_actual & set(ranked.head(10)['Name'].values)) > 0
        races += 1

    print(f"\n{'='*50}")
    print(f'Model: {name} | Races: {races}')
    print(f'  Top-1  accuracy: {top1/races*100:.1f}%')
    print(f'  Top-3  accuracy: {top3/races*100:.1f}%')
    print(f'  Top-5  accuracy: {top5/races*100:.1f}%')
    print(f'  Top-10 accuracy: {top10/races*100:.1f}%')
    return {'top1': top1/races, 'top3': top3/races,
            'top5': top5/races, 'top10': top10/races}


eval_df = test_df[['Race Name', 'tier', 'Name']].copy()

print('=== CLASSIFICATION METRICS ===')
results = {}
results['LightGBM'] = evaluate_model(
    'LightGBM', lgb_model, X_test, y_test
)
results['XGBoost'] = evaluate_model(
    'XGBoost', xgb_model, X_test, y_test,
    needs_imputation=True, medians=xgb_medians
)
results['Baseline'] = evaluate_model(
    'Logistic Regression', lr_pipeline, X_test, y_test
)

print('\n=== RANKING QUALITY (primary metric) ===')
ranking = {}
ranking['LightGBM'] = evaluate_ranking(
    'LightGBM', lgb_model, X_test, eval_df
)
ranking['XGBoost'] = evaluate_ranking(
    'XGBoost', xgb_model, X_test, eval_df,
    needs_imputation=True, medians=xgb_medians
)
ranking['Baseline'] = evaluate_ranking(
    'Logistic Regression', lr_pipeline, X_test, eval_df
)

print(f"\n{'='*55}")
print('SUMMARY')
print(f"{'Model':<25} {'LogLoss':>8} {'AUC':>6} {'Top1':>6} {'Top3':>6} {'Top5':>6} {'Top10':>7}")
print('-' * 65)
for name in ['LightGBM', 'XGBoost', 'Baseline']:
    r = results[name]
    k = ranking[name]
    print(f"  {name:<23} {r['log_loss']:>8.4f} {r['auc']:>6.4f} "
          f"{k['top1']*100:>5.1f}% {k['top3']*100:>5.1f}% "
          f"{k['top5']*100:>5.1f}% {k['top10']*100:>6.1f}%")

best_name = max(ranking, key=lambda k: ranking[k]['top5'])
print(f'\nBest model by top-5 ranking accuracy: {best_name}')


=== CLASSIFICATION METRICS ===

Model: LightGBM
  Log Loss: 0.6108
  AUC:      0.8171

  Per-class accuracy:
    winner     precision=0.29  recall=0.08  f1=0.12
    podium     precision=0.33  recall=0.02  f1=0.04
    top10      precision=0.33  recall=0.06  f1=0.10
    top20      precision=0.25  recall=0.04  f1=0.06
    finisher   precision=0.83  recall=0.98  f1=0.90
    dnf        precision=0.46  recall=0.16  f1=0.24

Model: XGBoost
  Log Loss: 0.6056
  AUC:      0.8219

  Per-class accuracy:
    winner     precision=0.20  recall=0.02  f1=0.03
    podium     precision=0.38  recall=0.02  f1=0.04
    top10      precision=0.30  recall=0.07  f1=0.11
    top20      precision=0.21  recall=0.02  f1=0.04
    finisher   precision=0.83  recall=0.98  f1=0.90
    dnf        precision=0.46  recall=0.17  f1=0.25

Model: Logistic Regression
  Log Loss: 0.6569
  AUC:      0.7848

  Per-class accuracy:
    winner     precision=0.17  recall=0.02  f1=0.03
    podium     precision=0.20  recall=0.02  f1=0.

In [22]:
importance_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print('=== FEATURE IMPORTANCE (XGBoost, by gain) ===\n')
max_imp = importance_df['importance'].max()
for _, row in importance_df.iterrows():
    bar = '█' * int(row['importance'] / max_imp * 45)
    print(f"  {row['feature']:<35} {bar}")


=== FEATURE IMPORTANCE (XGBoost, by gain) ===

  stage_num                           █████████████████████████████████████████████
  terrain_top10_rate                  ████████████████████████████████
  stage_type_enc                      ██████████████
  gc_proxy                            ██████████
  Cobblestones                        ██████████
  recent_avg_rank_5                   █████████
  terrain_avg_rank                    ████████
  terrain_podium_rate                 ████████
  recent_top10_rate_12mo              ███████
  vg_per_km                           ███████
  Vertical Gain                       ███████
  days_since_last_race                ██████
  recent_podium_rate_12mo             ██████
  Distance                            ██████
  cobble_pct                          ██████
  asphalt_pct                         ██████
  Highest Elevation                   ██████
  career_podium_rate                  █████
  terrain_win_rate                    █████
  terrain

In [23]:
def predict_stage(
    race_name: str,
    year: int,
    stage: int = None,
    top_n: int = 20,
) -> pd.DataFrame:
    """
    Predict finish tier probabilities for all starters
    in a given race/stage. Uses best model by top-5 accuracy.
    """
    # Use best model
    best_name  = max(ranking, key=lambda k: ranking[k]['top5'])
    best_model = lgb_model if best_name == 'LightGBM' else xgb_model
    needs_imp  = best_name == 'XGBoost'
    medians    = xgb_medians if needs_imp else None

    mask = (
        model_df['Race_results'].str.contains(
            race_name, na=False, case=False
        ) &
        (model_df['Year_results'] == year)
    )
    if stage is not None:
        mask &= (
            (model_df['Stage_results'] == stage) |
            (model_df['Stage_results'] == float(stage))
        )

    race_rows = model_df[mask].drop_duplicates('Name')

    if race_rows.empty:
        available = model_df[
            model_df['Race_results'].str.contains(
                race_name, na=False, case=False
            ) &
            (model_df['Year_results'] == year)
        ]['Stage_results'].dropna().unique()
        print(f'No data found for {race_name} {year}'
              + (f' Stage {stage}' if stage else ''))
        print(f'Available stages: {sorted(available)[:20]}')
        return pd.DataFrame()

    X_race = race_rows[FEATURE_COLS].copy()
    if needs_imp and medians is not None:
        X_race = X_race.fillna(medians)

    proba      = best_model.predict_proba(X_race)
    results_df = race_rows[['Name', 'Team', 'tier']].copy()
    results_df['actual_tier'] = race_rows['tier'].values

    for i, tier in enumerate(TIER_ORDER):
        results_df[f'p_{tier}'] = proba[:, i]

    return (
        results_df
        .sort_values('p_winner', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


# Test on TDF 2023 Stage 17
print('=== TDF 2023 Stage 17 Predictions ===')
print(f'(Using best model by top-5 accuracy)\n')

preds = predict_stage('Tour de France', 2023, stage=17)

if not preds.empty:
    print(f"{'Rider':<30} {'Team':<25} "
          f"{'Win%':>5} {'Pod%':>5} {'T10%':>5} "
          f"{'T20%':>5} {'Fin%':>5} {'DNF%':>5} "
          f"Actual")
    print('-' * 115)
    for _, row in preds.iterrows():
        marker = '◄' if row['actual_tier'] in ['winner','podium','top10'] else ''
        print(
            f"  {row['Name']:<28} {row['Team']:<23} "
            f"{row['p_winner']*100:5.1f} "
            f"{row['p_podium']*100:5.1f} "
            f"{row['p_top10']*100:5.1f} "
            f"{row['p_top20']*100:5.1f} "
            f"{row['p_finisher']*100:5.1f} "
            f"{row['p_dnf']*100:5.1f}  "
            f"{row['actual_tier']} {marker}"
        )


=== TDF 2023 Stage 17 Predictions ===
(Using best model by top-5 accuracy)

Rider                          Team                       Win%  Pod%  T10%  T20%  Fin%  DNF% Actual
-------------------------------------------------------------------------------------------------------------------
  VINGEGAARD Jonas             Jumbo-Visma              11.4  20.4  42.8  13.6   9.0   2.8  top10 ◄
  POGAČAR Tadej                UAE Team Emirates         9.6  16.6  49.2  10.6  10.4   3.6  finisher 
  PIDCOCK Thomas               INEOS Grenadiers          6.1   6.9  16.8  24.8  43.3   2.2  finisher 
  GALL Felix                   AG2R Citroën Team         5.7   5.2  29.5  32.6  24.6   2.4  winner ◄
  VAN AERT Wout                Jumbo-Visma               5.4   6.1  21.9  13.0  47.9   5.8  finisher 
  PINOT Thibaut                Groupama - FDJ            4.0  13.8  38.4  26.0  14.9   2.8  top20 
  YATES Adam                   UAE Team Emirates         4.0  13.8  51.3  20.2   7.7   3.1  top10 ◄
  

In [24]:
# Save the single best model artifact for use by the agent
best_name  = max(ranking, key=lambda k: ranking[k]['top5'])
print(f'Best Model: {best_name}')
best_model = lgb_model if best_name == 'LightGBM' else xgb_model

tier_predictor = {
    'model':              best_model,
    'model_name':         best_name,
    'feature_cols':       FEATURE_COLS,
    'tier_order':         TIER_ORDER,
    'tier_to_int':        TIER_TO_INT,
    'int_to_tier':        INT_TO_TIER,
    'stage_type_encoder': le_stage,
    'xgb_medians':        xgb_medians.to_dict(),
    'train_years':        list(range(2017, CUTOFF_YEAR)),
    'test_year':          CUTOFF_YEAR,
    'metrics': {
        name: {
            'log_loss': results[name]['log_loss'],
            'auc':      results[name]['auc'],
            'top1':     ranking[name]['top1'],
            'top3':     ranking[name]['top3'],
            'top5':     ranking[name]['top5'],
            'top10':    ranking[name]['top10'],
        }
        for name in results
    }
}

with open(MODELS_DIR / 'tier_predictor.pkl', 'wb') as f:
    pickle.dump(tier_predictor, f)

with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump(
        {name: {
            'log_loss': results[name]['log_loss'],
            'auc':      results[name]['auc'],
            'top5_ranking_accuracy': ranking[name]['top5'],
        } for name in results},
        f, indent=2
    )

print(f'Best model: {best_name}')
print(f'  Top-5 accuracy: {ranking[best_name]["top5"]*100:.1f}%')
print(f'  AUC:            {results[best_name]["auc"]:.4f}')
print(f'\nSaved: models/tier_predictor.pkl')
print(f'Saved: models/metrics.json')


Best Model: XGBoost
Best model: XGBoost
  Top-5 accuracy: 55.2%
  AUC:            0.8219

Saved: models/tier_predictor.pkl
Saved: models/metrics.json
